In [0]:
from pyspark.sql import functions as F

CATALOG = "automotive_project"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

labor_by_order = spark.sql(f"""
SELECT
    order_id,
    SUM(labor_hours) AS total_labor_hours,
    SUM(labor_cost) AS total_labor_cost
FROM {SILVER}.service_order_tasks
GROUP BY order_id
""")

parts_by_order = spark.sql(f"""
SELECT
    order_id,
    SUM(quantity) AS total_parts_quantity,
    SUM(total_part_cost) AS total_parts_cost
FROM {SILVER}.parts_used
GROUP BY order_id
""")

base = spark.sql(f"""
SELECT
    v.vehicle_id,
    v.vin,
    vm.make,
    vm.model_name,
    vm.body_type,
    vm.fuel_type,
    v.manufacturing_year,
    so.order_id
FROM {SILVER}.vehicles v
LEFT JOIN {SILVER}.vehicle_models vm
    ON v.model_id = vm.model_id
LEFT JOIN {SILVER}.service_appointments sa
    ON v.vehicle_id = sa.vehicle_id
LEFT JOIN {SILVER}.service_orders so
    ON sa.appointment_id = so.appointment_id
""")

vehicle_service = (
    base
    .join(labor_by_order, "order_id", "left")
    .join(parts_by_order, "order_id", "left")
    .groupBy(
        "vehicle_id",
        "vin",
        "make",
        "model_name",
        "body_type",
        "fuel_type",
        "manufacturing_year"
    )
    .agg(
        F.countDistinct("order_id").alias("total_service_orders"),
        F.sum(F.coalesce("total_labor_hours", F.lit(0))).alias("total_labor_hours"),
        F.sum(F.coalesce("total_labor_cost", F.lit(0))).alias("total_labor_cost"),
        F.sum(F.coalesce("total_parts_quantity", F.lit(0))).alias("total_parts_quantity"),
        F.sum(F.coalesce("total_parts_cost", F.lit(0))).alias("total_parts_cost")
    )
    .withColumn(
        "total_service_cost",
        F.col("total_labor_cost") + F.col("total_parts_cost")
    )
)

vehicle_service.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{GOLD}.vehicle_service_analysis")

print("✓ vehicle_service_analysis created")

display(
    spark.table(f"{GOLD}.vehicle_service_analysis")
    .orderBy(F.desc("total_service_cost"))
    .limit(20)
)

✓ vehicle_service_analysis created


vehicle_id,vin,make,model_name,body_type,fuel_type,manufacturing_year,total_service_orders,total_labor_hours,total_labor_cost,total_parts_quantity,total_parts_cost,total_service_cost
3737,VIN9IND2023A003737,Maruti Suzuki,Brezza,Compact SUV,Hybrid,2023,6,419.5999999999999,288105.0,496,1199350.0,1487455.0
3284,VIN9IND2024A003284,Toyota,Innova Hycross,MPV,Hybrid,2023,5,385.3,261675.0,425,1179200.0,1440875.0
5578,VIN9IND2023A005578,Tata,Nexon EV,Compact SUV,Electric,2021,5,349.0,247830.0,441,1180650.0,1428480.0
5186,VIN9IND2022A005186,Tata,Nexon EV,Compact SUV,Electric,2022,5,373.69999999999993,255270.0,433,1150450.0,1405720.0
4578,VIN9IND2023A004578,Tata,Harrier,Mid-size SUV,Diesel,2024,5,382.9,249540.0,403,1135800.0,1385340.0
7906,VIN9IND2023A007906,Mahindra,Thar,Off-road SUV,Diesel,2022,4,263.09999999999997,177735.0,408,1205250.0,1382985.0
7194,VIN9IND2023A007194,Mahindra,XUV700,Full SUV,Petrol,2023,5,321.9,213135.0,415,1149100.0,1362235.0
4144,VIN9IND2022A004144,Mahindra,Thar,Off-road SUV,Diesel,2021,5,307.7,202170.0,433,1154850.0,1357020.0
1432,VIN9IND2024A001432,Toyota,Fortuner,Full SUV,Diesel,2023,5,310.09999999999997,210465.0,422,1091200.0,1301665.0
8686,VIN9IND2024A008686,Hyundai,Ioniq 5,EV Crossover,Electric,2023,5,330.79999999999995,223290.0,406,1076850.0,1300140.0


In [0]:
# ============================================================
# WARRANTY CLAIM ANALYTICS
# ============================================================

warranty_analysis = spark.sql(f"""
SELECT
    wc.claim_id,
    wc.claim_number,
    wc.order_id,
    wc.claim_type,
    wc.submission_date,
    wc.claimed_amount,
    wc.approved_amount,
    wc.claim_status,

    CASE
        WHEN wc.claimed_amount > 0
        THEN ROUND(
            (wc.approved_amount / wc.claimed_amount) * 100,
            2
        )
        ELSE 0
    END AS approval_percentage,

    ROUND(
        wc.claimed_amount - wc.approved_amount,
        2
    ) AS rejected_amount

FROM {SILVER}.warranty_claims wc
""")

warranty_analysis.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{GOLD}.warranty_analysis")

print("✓ warranty_analysis created")

display(
    spark.table(f"{GOLD}.warranty_analysis")
    .orderBy(F.desc("claimed_amount"))
    .limit(20)
)

✓ warranty_analysis created


claim_id,claim_number,order_id,claim_type,submission_date,claimed_amount,approved_amount,claim_status,approval_percentage,rejected_amount
840,CLM-2024-10840,2097,Part Recall,2024-02-28,17986.0,0.0,PENDING_AUDIT,0.0,17986.0
641,CLM-2024-10641,2713,Extended Warranty,2024-02-15,17982.0,17982.0,APPROVED,100.0,0.0
1167,CLM-2024-11167,7012,Manufacturer Warranty,2024-03-22,17982.0,0.0,PENDING_AUDIT,0.0,17982.0
1762,CLM-2024-11762,4167,Part Recall,2024-05-02,17978.0,0.0,REJECTED,0.0,17978.0
1172,CLM-2024-11172,9867,Part Recall,2024-03-22,17978.0,0.0,REJECTED,0.0,17978.0
594,CLM-2024-10594,7227,Manufacturer Warranty,2024-02-11,17978.0,0.0,REJECTED,0.0,17978.0
1040,CLM-2024-11040,3512,Extended Warranty,2024-03-13,17973.0,0.0,PENDING_AUDIT,0.0,17973.0
1704,CLM-2024-11704,8210,Part Recall,2024-04-28,17967.0,0.0,REJECTED,0.0,17967.0
1813,CLM-2024-11813,9706,Extended Warranty,2024-05-05,17964.0,17964.0,APPROVED,100.0,0.0
1690,CLM-2024-11690,8435,Part Recall,2024-04-27,17943.0,17943.0,APPROVED,100.0,0.0


In [0]:
from pyspark.sql import functions as F

CATALOG = "automotive_project"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

# ============================================================
# DEALER PERFORMANCE ANALYSIS
# ============================================================

dealer_performance = spark.sql(f"""
WITH order_summary AS (
    SELECT
        sc.dealer_id,
        COUNT(DISTINCT so.order_id) AS total_service_orders,
        COUNT(DISTINCT CASE
            WHEN so.order_status = 'Completed'
            THEN so.order_id
        END) AS completed_orders
    FROM {SILVER}.service_orders so
    LEFT JOIN {SILVER}.service_appointments sa
        ON so.appointment_id = sa.appointment_id
    LEFT JOIN {SILVER}.service_centers sc
        ON sa.service_center_id = sc.service_center_id
    GROUP BY sc.dealer_id
),

labor_summary AS (
    SELECT
        sc.dealer_id,
        COALESCE(SUM(sot.labor_cost), 0) AS total_labor_cost
    FROM {SILVER}.service_order_tasks sot
    LEFT JOIN {SILVER}.service_orders so
        ON sot.order_id = so.order_id
    LEFT JOIN {SILVER}.service_appointments sa
        ON so.appointment_id = sa.appointment_id
    LEFT JOIN {SILVER}.service_centers sc
        ON sa.service_center_id = sc.service_center_id
    GROUP BY sc.dealer_id
),

parts_summary AS (
    SELECT
        sc.dealer_id,
        COALESCE(SUM(pu.total_part_cost), 0) AS total_parts_cost
    FROM {SILVER}.parts_used pu
    LEFT JOIN {SILVER}.service_orders so
        ON pu.order_id = so.order_id
    LEFT JOIN {SILVER}.service_appointments sa
        ON so.appointment_id = sa.appointment_id
    LEFT JOIN {SILVER}.service_centers sc
        ON sa.service_center_id = sc.service_center_id
    GROUP BY sc.dealer_id
),

center_summary AS (
    SELECT
        dealer_id,
        COUNT(DISTINCT service_center_id) AS service_center_count
    FROM {SILVER}.service_centers
    GROUP BY dealer_id
),

technician_summary AS (
    SELECT
        sc.dealer_id,
        COUNT(DISTINCT t.technician_id) AS technician_count
    FROM {SILVER}.technicians t
    LEFT JOIN {SILVER}.service_centers sc
        ON t.service_center_id = sc.service_center_id
    GROUP BY sc.dealer_id
)

SELECT
    d.dealer_id,
    d.dealer_name,
    cs.service_center_count,
    ts.technician_count,
    COALESCE(os.total_service_orders, 0) AS total_service_orders,
    COALESCE(os.completed_orders, 0) AS completed_orders,
    COALESCE(ls.total_labor_cost, 0) AS total_labor_cost,
    COALESCE(ps.total_parts_cost, 0) AS total_parts_cost,
    COALESCE(ls.total_labor_cost, 0)
        + COALESCE(ps.total_parts_cost, 0) AS total_service_revenue
FROM {SILVER}.dealers d
LEFT JOIN center_summary cs
    ON d.dealer_id = cs.dealer_id
LEFT JOIN technician_summary ts
    ON d.dealer_id = ts.dealer_id
LEFT JOIN order_summary os
    ON d.dealer_id = os.dealer_id
LEFT JOIN labor_summary ls
    ON d.dealer_id = ls.dealer_id
LEFT JOIN parts_summary ps
    ON d.dealer_id = ps.dealer_id
ORDER BY total_service_revenue DESC
""")

# Write to Gold as Delta table
dealer_performance.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{GOLD}.dealer_performance")

print("✓ dealer_performance created")

display(
    spark.table(f"{GOLD}.dealer_performance")
)

✓ dealer_performance created


dealer_id,dealer_name,service_center_count,technician_count,total_service_orders,completed_orders,total_labor_cost,total_parts_cost,total_service_revenue
12,Apex Mobility Dealer 12,6,26,1222,0,5.572383E7,2.489435E8,3.0466733E8
6,Apex Mobility Dealer 6,6,20,1204,0,5.4788925E7,2.450753E8,2.99864225E8
11,Apex Mobility Dealer 11,5,16,977,0,4.432536E7,1.968292E8,2.4115456E8
2,Apex Mobility Dealer 2,4,17,829,0,3.7649295E7,1.6770995E8,2.05359245E8
4,Apex Mobility Dealer 4,4,17,815,0,3.7113885E7,1.6320885E8,2.00322735E8
8,Apex Mobility Dealer 8,4,10,795,0,3.600555E7,1.624289E8,1.9843445E8
5,Apex Mobility Dealer 5,4,21,777,0,3.5251665E7,1.568931E8,1.92144765E8
7,Apex Mobility Dealer 7,3,9,620,0,2.8680615E7,1.2519435E8,1.53874965E8
9,Apex Mobility Dealer 9,3,11,611,0,2.7431025E7,1.2400745E8,1.51438475E8
18,Apex Mobility Dealer 18,2,17,423,0,1.928484E7,8.576715E7,1.0505199E8


In [0]:
from pyspark.sql import functions as F

CATALOG = "automotive_project"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

# Aggregate labor costs separately
labor = (
    spark.table(f"{SILVER}.service_order_tasks")
    .groupBy("order_id")
    .agg(
        F.sum("labor_hours").alias("total_labor_hours"),
        F.sum("labor_cost").alias("total_labor_cost")
    )
)

# Aggregate parts costs separately
parts = (
    spark.table(f"{SILVER}.parts_used")
    .groupBy("order_id")
    .agg(
        F.sum("quantity").alias("total_parts_quantity"),
        F.sum("total_part_cost").alias("total_parts_cost")
    )
)

# Service order + service type
orders = (
    spark.table(f"{SILVER}.service_orders").alias("so")
    .join(
        spark.table(f"{SILVER}.service_appointments").alias("sa"),
        F.col("so.appointment_id") == F.col("sa.appointment_id"),
        "left"
    )
    .join(
        spark.table(f"{SILVER}.service_types").alias("st"),
        F.col("sa.service_type_id") == F.col("st.service_type_id"),
        "left"
    )
    .select(
        F.col("so.order_id"),
        F.col("so.order_status"),
        F.col("so.check_in_time"),
        F.col("so.completion_time"),
        F.col("st.service_type_id"),
        F.col("st.service_name")
    )
)

# Combine aggregated costs
service_cost = (
    orders
    .join(labor, "order_id", "left")
    .join(parts, "order_id", "left")
    .fillna(0, subset=[
        "total_labor_hours",
        "total_labor_cost",
        "total_parts_quantity",
        "total_parts_cost"
    ])
    .withColumn(
        "total_service_cost",
        F.col("total_labor_cost") + F.col("total_parts_cost")
    )
)

# Save Gold table
(
    service_cost.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(f"{GOLD}.service_cost_analysis")
)

print("✓ service_cost_analysis created")

display(
    spark.table(f"{GOLD}.service_cost_analysis")
    .orderBy(F.desc("total_service_cost"))
    .limit(20)
)

✓ service_cost_analysis created


order_id,order_status,check_in_time,completion_time,service_type_id,service_name,total_labor_hours,total_labor_cost,total_parts_quantity,total_parts_cost,total_service_cost
6433,CLOSED,2024-06-05T07:31:03.000Z,2024-06-05T16:31:03.000Z,3,Brake System Overhaul,84.59999999999998,55125.0,100,361550.0,416675.0
136,CLOSED,2024-01-04T06:44:23.000Z,2024-01-04T15:44:23.000Z,2,Periodic Maintenance Service (PMS),66.1,44460.0,96,362800.0,407260.0
8548,CLOSED,2024-07-26T17:06:32.000Z,2024-07-27T06:06:32.000Z,3,Brake System Overhaul,52.8,37695.0,105,367250.0,404945.0
7968,CLOSED,2024-07-12T14:49:08.000Z,2024-07-13T08:49:08.000Z,6,Engine Tuning & Diagnostic Scan,68.0,47085.0,118,355800.0,402885.0
8340,CLOSED,2024-07-21T15:47:28.000Z,2024-07-22T08:47:28.000Z,4,Suspension & Steering Overhaul,66.7,46170.0,113,356000.0,402170.0
9482,CLOSED,2024-08-18T09:52:19.000Z,2024-08-18T17:52:19.000Z,6,Engine Tuning & Diagnostic Scan,86.19999999999999,60630.0,111,339550.0,400180.0
9182,CLOSED,2024-08-11T02:53:40.000Z,2024-08-11T17:53:40.000Z,1,First Free Checkup,78.70000000000002,53670.0,122,342650.0,396320.0
9017,CLOSED,2024-08-07T02:39:25.000Z,2024-08-07T14:39:25.000Z,2,Periodic Maintenance Service (PMS),54.00000000000001,32805.0,113,363050.0,395855.0
1592,CLOSED,2024-02-08T15:57:50.000Z,2024-02-09T12:57:50.000Z,1,First Free Checkup,62.69999999999999,42840.0,120,351400.0,394240.0
2575,CLOSED,2024-03-03T13:18:24.000Z,2024-03-04T02:18:24.000Z,5,Electrical Diagnostic,76.79999999999998,50595.0,104,343350.0,393945.0


In [0]:
from pyspark.sql import functions as F

CATALOG = "automotive_project"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

# ─────────────────────────────────────────────────────────────
# Technician Performance Analysis
# ─────────────────────────────────────────────────────────────

technician_performance = spark.sql(f"""
WITH order_metrics AS (
    SELECT
        assigned_technician_id AS technician_id,
        COUNT(DISTINCT order_id) AS total_service_orders,
        COUNT(DISTINCT CASE
            WHEN order_status = 'Completed'
            THEN order_id
        END) AS completed_orders
    FROM {SILVER}.service_orders
    WHERE assigned_technician_id IS NOT NULL
    GROUP BY assigned_technician_id
),

task_metrics AS (
    SELECT
        so.assigned_technician_id AS technician_id,
        COALESCE(SUM(sot.labor_hours), 0) AS total_labor_hours,
        COALESCE(SUM(sot.labor_cost), 0) AS total_labor_cost,
        COALESCE(AVG(sot.labor_hours), 0) AS avg_labor_hours
    FROM {SILVER}.service_orders so
    LEFT JOIN {SILVER}.service_order_tasks sot
        ON so.order_id = sot.order_id
    WHERE so.assigned_technician_id IS NOT NULL
    GROUP BY so.assigned_technician_id
)

SELECT
    t.technician_id,
    t.first_name,
    t.last_name,
    t.qualification,
    t.service_center_id,
    t.hourly_rate,

    COALESCE(om.total_service_orders, 0) AS total_service_orders,
    COALESCE(om.completed_orders, 0) AS completed_orders,

    COALESCE(tm.total_labor_hours, 0) AS total_labor_hours,
    COALESCE(tm.total_labor_cost, 0) AS total_labor_cost,
    COALESCE(tm.avg_labor_hours, 0) AS avg_labor_hours,

    CASE
        WHEN COALESCE(om.total_service_orders, 0) > 0
        THEN ROUND(
            om.completed_orders * 100.0 /
            om.total_service_orders,
            2
        )
        ELSE 0
    END AS completion_rate

FROM {SILVER}.technicians t
LEFT JOIN order_metrics om
    ON t.technician_id = om.technician_id
LEFT JOIN task_metrics tm
    ON t.technician_id = tm.technician_id
""")

# Save as Gold Delta table
technician_performance.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{GOLD}.technician_performance")

print("✓ technician_performance created")

display(
    spark.table(f"{GOLD}.technician_performance")
    .orderBy(F.desc("completion_rate"))
    .limit(20)
)

✓ technician_performance created


technician_id,first_name,last_name,qualification,service_center_id,hourly_rate,total_service_orders,completed_orders,total_labor_hours,total_labor_cost,avg_labor_hours,completion_rate
85,Karthik,Kumar,Master Technician,17,900.0,41,0,2791.2999999999965,1909965.0,1.3471525096525079,0.00
173,Priya,Nair,Apprentice,13,450.0,50,0,3480.099999999994,2370660.0,1.36207436399217,0.00
51,Karthik,Kumar,Apprentice,38,450.0,42,0,2833.6999999999953,1902765.0,1.3303755868544578,0.00
61,Deepa,Menon,Service Tech Level 2,47,600.0,53,0,3575.9999999999927,2409495.0,1.3659281894575985,0.00
134,Sneha,Reddy,Senior Diagnostic Specialist,5,750.0,44,0,2962.7999999999956,2002440.0,1.352259242355087,0.00
160,Deepa,Nair,Senior Diagnostic Specialist,18,750.0,39,0,2669.7999999999965,1808550.0,1.3315710723192002,0.00
133,Vikram,Menon,Apprentice,33,450.0,59,0,3923.799999999991,2664450.0,1.3400956284152976,0.00
49,Arun,Kumar,Senior Diagnostic Specialist,50,750.0,46,0,3099.399999999994,2085900.0,1.3605794556628596,0.00
99,Rohan,Reddy,Apprentice,11,450.0,53,0,3460.299999999993,2328315.0,1.3607156901297652,0.00
97,Vikram,Menon,Apprentice,11,450.0,61,0,3998.6999999999903,2679585.0,1.3445527908540653,0.00


In [0]:
from pyspark.sql import functions as F

service_center_performance = spark.sql(f"""
WITH order_metrics AS (
    SELECT
        sa.service_center_id,

        COUNT(DISTINCT so.order_id) AS total_service_orders,

        COUNT(DISTINCT CASE
            WHEN UPPER(TRIM(so.order_status)) IN ('COMPLETED', 'CLOSED')
            THEN so.order_id
        END) AS completed_orders,

        COALESCE(SUM(sot.labor_cost), 0) AS total_labor_cost

    FROM {SILVER}.service_orders so

    LEFT JOIN {SILVER}.service_appointments sa
        ON so.appointment_id = sa.appointment_id

    LEFT JOIN {SILVER}.service_order_tasks sot
        ON so.order_id = sot.order_id

    GROUP BY sa.service_center_id
),

technician_metrics AS (
    SELECT
        service_center_id,
        COUNT(DISTINCT technician_id) AS technician_count
    FROM {SILVER}.technicians
    GROUP BY service_center_id
)

SELECT
    sc.service_center_id,
    sc.center_name,
    sc.dealer_id,
    sc.total_bays,

    COALESCE(tm.technician_count, 0) AS technician_count,

    COALESCE(om.total_service_orders, 0) AS total_service_orders,

    COALESCE(om.completed_orders, 0) AS completed_orders,

    COALESCE(om.total_labor_cost, 0) AS total_labor_cost,

    CASE
        WHEN COALESCE(om.total_service_orders, 0) > 0
        THEN ROUND(
            om.completed_orders * 100.0 /
            om.total_service_orders,
            2
        )
        ELSE 0
    END AS completion_rate,

    CASE
        WHEN COALESCE(om.total_service_orders, 0) > 0
        THEN ROUND(
            om.total_labor_cost /
            om.total_service_orders,
            2
        )
        ELSE 0
    END AS average_labor_cost

FROM {SILVER}.service_centers sc

LEFT JOIN order_metrics om
    ON sc.service_center_id = om.service_center_id

LEFT JOIN technician_metrics tm
    ON sc.service_center_id = tm.service_center_id

ORDER BY total_service_orders DESC
""")

service_center_performance.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{GOLD}.service_center_performance")

print("✓ service_center_performance created")

display(
    spark.table(f"{GOLD}.service_center_performance")
)

✓ service_center_performance created


service_center_id,center_name,dealer_id,total_bays,technician_count,total_service_orders,completed_orders,total_labor_cost,completion_rate,average_labor_cost
49,Workshop Hub Bay-49,12,12,1,226,226,1.0241145E7,100.00,45314.8
25,Workshop Hub Bay-25,9,6,3,225,225,1.017786E7,100.00,45234.93
16,Workshop Hub Bay-16,18,6,8,221,221,1.015968E7,100.00,45971.4
27,Workshop Hub Bay-27,12,12,5,220,220,1.0169325E7,100.00,46224.2
33,Workshop Hub Bay-33,2,6,4,220,220,1.014495E7,100.00,46113.41
1,Workshop Hub Bay-1,10,6,2,215,215,9790545.0,100.00,45537.42
13,Workshop Hub Bay-13,4,12,7,215,215,9825150.0,100.00,45698.37
37,Workshop Hub Bay-37,4,8,5,214,214,9681255.0,100.00,45239.51
48,Workshop Hub Bay-48,7,10,4,214,214,9945435.0,100.00,46474.0
39,Workshop Hub Bay-39,8,12,2,214,214,9506130.0,100.00,44421.17
